In [ ]:
import os
import psycopg
from age.networkx import *
from age import *
import networkx as nx

conn = psycopg.connect(
    host=os.environ.get('PGHOST', 'localhost'),
    port=os.environ.get('PGPORT', '5432'),
    dbname=os.environ.get('PGDATABASE', 'postgres'),
    user=os.environ.get('PGUSER', 'postgres'),
    password=os.environ.get('PGPASSWORD', ''))
graphName = 'age_networkx_sample'


age.setUpAge(conn, graphName)

### Creating a Random Networkx Graph

In [2]:
import random
num_nodes = 5
num_edges = 10
G = nx.DiGraph()
try:
    for i in range(num_nodes//2):
        G.add_node(i, label='Number',properties={'name' : i})
    for i in range(num_nodes//2,num_nodes):
        G.add_node(i, label='Integer',properties={'age' : i*2} )
    for i in range(num_edges):
        source = random.randint(0, num_nodes-1)
        target = random.randint(0, num_nodes-1)
        G.add_edge(source, target, label='Connection' ,properties={'st' : source , 'ed':target})
except Exception as e:
    raise e
print(G)

DiGraph with 5 nodes and 9 edges


## Networkx to AGE

In [3]:
networkx_to_age(conn, G, graphName)

## AGE to Networkx

### ALL AGE to Networkx


In [4]:
G = age_to_networkx(conn, graphName)
print(G)

DiGraph with 5 nodes and 9 edges


### Add subgraph of AGE to Networkx using query

In [5]:
G = age_to_networkx(conn, graphName, 
                    query="""
SELECT * from cypher('%s', $$
        MATCH (V:Integer)
        RETURN V
$$) as (V agtype);
""" % graphName)

G = age_to_networkx(conn, graphName, G=G,
                    query="""
SELECT * from cypher('%s', $$
        MATCH (V:Number)
        RETURN V
$$) as (V agtype);
""" % graphName)

G = age_to_networkx(conn, graphName, G=G,
                    query="""
SELECT * from cypher('%s', $$
        MATCH (V)-[R]->(V2)
        RETURN V,R,V2
$$) as (V agtype, R agtype, V2 agtype);
""" % graphName)
print(G)


DiGraph with 5 nodes and 9 edges
